# 📊 Employee Performance Predictor: Exploratory Data Analysis (EDA)

**Project**: Employee Performance Predictor using Data Analytics  
**Phase**: Phase 4 — Exploratory Data Analysis  
**Dataset**: `data/synthetic/employee_performance.csv` (1,000 employees × 4 quarters = 4,000 observations)  

> **Important Note**: All data analyzed herein is synthetically generated for simulation and machine learning methodology demonstration. The relationships observed are mathematical properties of the simulation model and do not represent empirical observations of real workforce behavior.

## 1. Environment Setup & Data Ingestion
We begin by importing necessary libraries and loading the synthetic dataset.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Add project src directory to sys.path
PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from employee_performance.eda import (
    compute_dataset_overview,
    compute_correlations,
    compute_department_metrics,
    compute_job_role_metrics,
    compute_period_trends,
    analyze_training_relationship,
    NUMERIC_PREDICTORS,
    FEATURE_SPECS,
)

# Load synthetic dataset
DATA_PATH = PROJECT_ROOT / "data" / "synthetic" / "employee_performance.csv"
df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

## 2. Dataset Overview & Data Quality Verification
Let's inspect the dimensions, completeness, and parametric summaries across all attributes.

In [ ]:
overview = compute_dataset_overview(df)
print(f"Unique Employees: {overview['unique_employees']:,}")
print(f"Total Missing Values: {sum(overview['missing_counts'].values())}")
print(f"Exact Duplicate Rows: {overview['duplicate_rows']}")
print(f"Duplicate Composite Keys (employee_id, period): {overview['duplicate_keys']}")

# Numerical Summary Table
num_df = pd.DataFrame(overview['numeric_stats']).T[['count', 'mean', 'std', 'min', 'median', 'max', 'skewness']]
num_df.round(2)

## 3. Target Distribution (`future_performance_score`)
We inspect the continuous rating scale ($1.00$ to $5.00$) to check for normality, skewness, or ceiling/floor clipping effects.

In [ ]:
fig, (ax_box, ax_hist) = plt.subplots(2, 1, figsize=(8, 5), sharex=True, gridspec_kw={'height_ratios': [0.25, 0.75]})
target = df['future_performance_score']
ax_box.boxplot(target, vert=False, patch_artist=True, boxprops=dict(facecolor='#3498db'))
ax_box.set_yticks([])
ax_box.set_title('Target Distribution: Future Performance Score')

ax_hist.hist(target, bins=25, color='#5dade2', edgecolor='#1f618d', density=True, alpha=0.75)
ax_hist.axvline(target.mean(), color='#e74c3c', linestyle='--', label=f'Mean: {target.mean():.2f}')
ax_hist.axvline(target.median(), color='#27ae60', linestyle=':', label=f'Median: {target.median():.2f}')
ax_hist.set_xlabel('Future Performance Score [1.0 - 5.0]')
ax_hist.set_ylabel('Density')
ax_hist.legend()
plt.tight_layout()
plt.show()

**Interpretation**:
The target exhibits a symmetric, bell-shaped distribution centered at mean **3.24** and median **3.25** with standard deviation **0.64**. Minimal skewness (-0.07) indicates healthy spread without degenerate collapse onto boundaries.

## 4. Correlation Analysis & Feature Associations
We calculate Pearson and Spearman rank correlations between all numeric predictors and the future performance rating.

In [ ]:
corr_results = compute_correlations(df)
corr_df = pd.DataFrame({
    'Pearson_r': corr_results['target_pearson'],
    'Spearman_rho': corr_results['target_spearman']
})
corr_df.round(4)

## 5. Investigating the Training Hours Paradox (Selection Bias)
Notice that `training_hours_previous_period` has a negative marginal correlation ($r = -0.2791$) with performance. Let's stratify by prior performance tier to understand this mechanism.

In [ ]:
train_res = analyze_training_relationship(df)
pd.DataFrame(train_res['tier_analysis']).T

**Interpretation**:
Struggling employees receive an average of **14.9 hours** of mandated remedial training, whereas high performers average only **10.6 hours**. This is a classic case of **confounding by indication**: struggling performers are assigned training *because* of poor performance, inducing a negative raw correlation even though training's structural conditional effect is positive.

## 6. Quarterly Trends & Within-Employee Dynamics
Because this is a longitudinal panel with repeated observations, we evaluate quarter-over-quarter transitions.

In [ ]:
trends = compute_period_trends(df)
print('Macro Period Summary:')
print(pd.DataFrame(trends['period_summary']).T)
print('\nWithin-Employee Transition Summary (n=3,000 transitions):')
for k, v in trends['within_employee_metrics'].items():
    print(f'  {k}: {v}')

## 7. Repeated-Measures Awareness & Methodological Guidance
- **Clustered Panel Data**: The 4,000 rows represent 1,000 unique employees tracked over 4 quarters. They are not 4,000 i.i.d. observations.
- **Model Validation Design**: When partitioning data for model training and evaluation (Phases 5 & 6), we must employ **GroupKFold (grouped on `employee_id`)** or **Temporal Out-of-Time split** (e.g. Quarters 1–3 train, Quarter 4 test) to prevent data leakage and avoid overestimating model generalization.